# Massa-Mola-Amortecedor — Sistema de 2ª ordem

Notebook complementar às aulas [CN07 — EDOs, Euler e Runge-Kutta](../Aulas/CN07EDOsEuler.qmd)
e [CN08 — EDOs, Sistemas e Estabilidade](../Aulas/CN08EDOsSistemas.qmd).

Sistema massa-mola-amortecedor forçado:

$$m\ddot{x} + c\dot{x} + kx = F(t)$$

Reduzido a um sistema de 1ª ordem com $x_1 = x,\ x_2 = \dot{x}$ e integrado por dois métodos:
**Euler explícito** (passo fixo) e **`solve_ivp`/RK45** (passo adaptativo), para comparar precisão.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

## Parâmetros e força de excitação

In [ ]:
m = 1.0   # massa (kg)
c = 0.4   # amortecimento (N.s/m)
k = 10.0  # rigidez da mola (N/m)

# Força em degrau, aplicada a partir de t = 1 s
F = lambda t: 1.0 * (t > 1.0)

In [ ]:
plt.figure(figsize=(6, 2))
t_plot = np.linspace(0, 15, 3000)
plt.step(t_plot, [F(ti) for ti in t_plot])
plt.title('Força de excitação F(t)')
plt.xlabel('t (s)')
plt.grid()
plt.show()

## Sistema de estado

$$\begin{cases} \dot{x}_1 = x_2 \\ \dot{x}_2 = \dfrac{1}{m}\bigl(F(t) - c\,x_2 - k\,x_1\bigr) \end{cases}$$

In [ ]:
def sistema(t, x):
    x1, x2 = x
    dx1 = x2
    dx2 = (F(t) - c * x2 - k * x1) / m
    return np.array([dx1, dx2])

## Integração — Euler explícito (passo fixo)

In [ ]:
t = np.linspace(0, 15, 3000)
Ts = t[1] - t[0]

X_euler = np.zeros((len(t), 2))
for n in range(len(t) - 1):
    X_euler[n + 1] = X_euler[n] + Ts * sistema(t[n], X_euler[n])

## Integração — `solve_ivp` (RK45, passo adaptativo)

In [ ]:
sol = solve_ivp(sistema, [t[0], t[-1]], [0.0, 0.0], method='RK45',
                 t_eval=t, rtol=1e-8, atol=1e-10)
X_rk45 = sol.y.T

In [ ]:
plt.figure(figsize=(6, 4))

plt.subplot(211)
plt.plot(t, X_euler[:, 0], label='Euler')
plt.plot(t, X_rk45[:, 0], '--', label='RK45 (solve_ivp)')
plt.ylabel('posição $x$')
plt.legend()
plt.grid()

plt.subplot(212)
plt.plot(t, X_euler[:, 1], label='Euler')
plt.plot(t, X_rk45[:, 1], '--', label='RK45 (solve_ivp)')
plt.ylabel('velocidade $\dot{x}$')
plt.xlabel('t (s)')
plt.grid()

plt.tight_layout()
plt.show()

::: {.callout-tip}
Com o mesmo passo $T_s$, o Euler explícito acumula erro perceptível na amplitude/fase da oscilação,
enquanto o RK45 (passo adaptativo, erro de 4ª/5ª ordem) acompanha a solução de referência —
o mesmo contraste discutido na aula CN07 (ordem de convergência) e CN08 (estabilidade).
:::

## Polos do sistema — ligação com estabilidade (CN08)

In [ ]:
den = np.array([m, c, k])   # m*s^2 + c*s + k
polos = np.roots(den)
print('Polos:', polos)

wn = np.sqrt(k / m)                      # frequência natural
zeta = c / (2 * np.sqrt(k * m))          # razão de amortecimento
print(f'Frequência natural wn = {wn:.4f} rad/s')
print(f'Razão de amortecimento zeta = {zeta:.4f}  ({"sub" if zeta < 1 else "sobre"}amortecido)')